In [1]:
import pandas as pd

INPUT_PATH = "bat_posts_results_final_patched.csv"   # <-- edit path if needed

df = pd.read_csv(INPUT_PATH)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")

Shape: 144652 rows x 16 columns


In [2]:
n_positive = (df["bat_score"] > 0).sum()

print(f"bat_score >  0  : {n_positive} ({n_positive/len(df)*100:.1f}%)")


bat_score >  0  : 12237 (8.5%)


In [3]:
# 2. FILTER the actual rows (don't just save the True/False mask)
filtered_df = df[df["bat_score"] > 0]

# 3. Save the filtered rows to the CSV
filtered_df.to_csv('bat_score_pos.csv', index=False)
print("done")

done


In [4]:
print(f"Shape: {filtered_df.shape[0]} rows x {filtered_df.shape[1]} columns")

Shape: 12237 rows x 16 columns


In [5]:
"""
PHASE 2A→COMMENTS — FILTER + PREPROCESS TOP-LEVEL COMMENTS
=============================================================
Goal: from raw per-subreddit comment JSON dumps, keep ONLY
top-level comments belonging to posts with bat_score > 0,
then clean them using the same clean_text() logic as Phase 0.

Why filter-first: raw comment dumps are typically 10-50x the
size of the post dumps. Cleaning everything before filtering
wastes most of that work. This filters on raw JSON first
(cheap: just checking two ID fields), THEN cleans only the
much smaller relevant subset.

Reads:
    BAT_SCORES_FILE            (to build target post_id set)
    r_<subreddit>_comments.json  (raw, per subreddit; NDJSON or JSON array)

Writes (per subreddit, checkpointed):
    processed_subreddits/filtered_raw_<subreddit>_comments.csv   (filtered, uncleaned)
Writes (final):
    master_comments_filtered.csv   (all subreddits, cleaned, top-level only)
    comment_filter_summary.csv
"""

import json
import os
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

# ── CONFIGURATION ────────────────────────────────────────────────────────────
INPUT_DIR   = '/Users/nadia/Desktop/redditRun_june/comment_data/'   # where r_<sub>_comments.jsonl live
OUTPUT_DIR  = '/Users/nadia/Desktop/redditRun_june/comment_data/'
PROCESSED_DATA_DIR = OUTPUT_DIR + 'processed_subreddits/'
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

BAT_SCORES_FILE = OUTPUT_DIR + 'bat_score_pos.csv'   # pre-filtered bat_score>0 posts, one row per post
RAW_COMMENT_EXT = '.jsonl'   # confirmed NDJSON — one comment object per line

SUBREDDITS = ['ciso', 'cybersecurity', 'SecurityCareerAdvice', 'sysadmin', 'asknetsec']

START_DATE = '2018-01-01'
END_DATE   = '2026-04-26'
MIN_COMMENT_WORDS = 10   # same threshold you used for posts pipeline

# Fields to keep from raw comment JSON (Reddit API / PRAW-style naming —
# adjust here if your scraper used different field names, e.g. 'selftext' vs 'body')
COMMENT_FIELDS = ['id', 'author', 'body', 'created_utc', 'score',
                   'link_id', 'parent_id', 'permalink', 'subreddit']


# ── TEXT CLEANING (same as Phase 0) ─────────────────────────────────────────
def clean_text(text):
    if pd.isna(text) or text == "":
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', ' URL ', text, flags=re.MULTILINE)
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    text = re.sub(r'\*\*([^\*]+)\*\*', r'\1', text)
    text = re.sub(r'\*([^\*]+)\*', r'\1', text)
    text = re.sub(r'~~([^~]+)~~', r'\1', text)
    text = re.sub(r'#{1,6}\s', '', text)
    text = re.sub(r'r/\w+', ' SUBREDDIT ', text)
    text = re.sub(r'u/\w+', ' USER ', text)
    text = re.sub(r'\[([^\]]*)\]', '', text)
    text = re.sub(r'\(([^\)]*)\)', '', text)
    text = re.sub(r'\{([^\}]*)\}', '', text)
    text = re.sub(r'[^\w\s\.\-_:/]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[^a-zA-Z\s\.\,\!\?\;\:]', ' ', text)
    text = text.lower()
    return text


# ── STEP 1: BUILD TARGET POST_ID SET ────────────────────────────────────────
def load_target_post_ids():
    df = pd.read_csv(BAT_SCORES_FILE, usecols=['post_id', 'bat_score'])
    target = df.loc[df['bat_score'] > 0, 'post_id'].astype(str)
    target_set = set(target)
    print(f"Target posts (bat_score>0): {len(target_set):,} of {len(df):,} total")
    return target_set


# ── STEP 2: DETECT JSON FORMAT ──────────────────────────────────────────────
def detect_json_format(filepath):
    """Peek at the file to see if it's NDJSON (one object/line) or a JSON array."""
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            stripped = line.strip()
            if not stripped:
                continue
            return 'array' if stripped[0] == '[' else 'ndjson'
    return 'ndjson'


# ── STEP 3: STREAM-FILTER RAW COMMENTS (memory-safe, no full-file load) ────
def stream_filter_comments(filepath, target_post_ids, fmt):
    """
    Yields dicts for comments that are:
      (a) top-level (parent_id starts with 't3_', i.e. parent is the post itself)
      (b) belong to a post in target_post_ids (link_id, 't3_' prefix stripped)
    Streams the file rather than loading it whole.
    """
    kept = 0
    scanned = 0

    if fmt == 'ndjson':
        with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                scanned += 1
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                row = _check_and_extract(rec, target_post_ids)
                if row is not None:
                    kept += 1
                    yield row
    else:  # 'array' format — try ijson for true streaming, fall back to json.load
        try:
            import ijson
            with open(filepath, 'rb') as f:
                for rec in ijson.items(f, 'item'):
                    scanned += 1
                    row = _check_and_extract(rec, target_post_ids)
                    if row is not None:
                        kept += 1
                        yield row
        except ImportError:
            print("  ⚠ ijson not installed — falling back to full json.load "
                  "(pip install ijson --break-system-packages for large files)")
            with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
                data = json.load(f)
            for rec in data:
                scanned += 1
                row = _check_and_extract(rec, target_post_ids)
                if row is not None:
                    kept += 1
                    yield row

    print(f"  Scanned {scanned:,} raw comments → kept {kept:,} "
          f"top-level comments on target posts")


def _check_and_extract(rec, target_post_ids):
    link_id = str(rec.get('link_id', ''))
    parent_id = str(rec.get('parent_id', ''))

    # top-level check: parent of comment is the post (t3_), not another comment (t1_)
    if not parent_id.startswith('t3_'):
        return None

    post_id = link_id.replace('t3_', '', 1)
    if post_id not in target_post_ids:
        return None

    row = {k: rec.get(k) for k in COMMENT_FIELDS}
    row['post_id'] = post_id
    return row


# ── STEP 4: PER-SUBREDDIT FILTER PASS (checkpointed) ────────────────────────
def filter_subreddit(subreddit_name, target_post_ids):
    print(f"\nFiltering comments for r/{subreddit_name}…")
    raw_path = INPUT_DIR + f'r_{subreddit_name}_comments{RAW_COMMENT_EXT}'

    if not os.path.exists(raw_path):
        print(f"  Skipping r/{subreddit_name}: file not found at {raw_path}")
        return pd.DataFrame()

    fmt = detect_json_format(raw_path)
    print(f"  Detected format: {fmt}")

    rows = list(stream_filter_comments(raw_path, target_post_ids, fmt))
    filtered = pd.DataFrame(rows)

    checkpoint_path = PROCESSED_DATA_DIR + f'filtered_raw_{subreddit_name}_comments.csv'
    filtered.to_csv(checkpoint_path, index=False)
    print(f"  ✓ Checkpoint saved: {checkpoint_path}")
    return filtered


# ── STEP 5: CLEAN THE FILTERED (SMALL) SET ──────────────────────────────────
def clean_filtered_comments(comments):
    if comments.empty:
        return comments

    before = len(comments)
    comments = comments.drop_duplicates(subset=['id'])
    print(f"  Removed {before - len(comments):,} duplicate comments")

    comments['created_date'] = pd.to_datetime(comments['created_utc'], unit='s', errors='coerce')
    comments['year']       = comments['created_date'].dt.year
    comments['month']      = comments['created_date'].dt.month
    comments['year_month'] = comments['created_date'].dt.to_period('M').astype(str)

    before = len(comments)
    comments = comments[comments['created_date'].notna()]
    comments = comments[
        (comments['created_date'] >= START_DATE) &
        (comments['created_date'] <= END_DATE)
    ]
    print(f"  Date window filter: kept {len(comments):,} (removed {before - len(comments):,})")

    comments['body'] = comments['body'].astype(str)
    before = len(comments)
    comments = comments[~comments['body'].isin(['[deleted]', '[removed]', 'nan', ''])]
    print(f"  Removed {before - len(comments):,} deleted/removed comments")

    comments['cleaned_body'] = comments['body'].apply(clean_text)
    comments['word_count']   = comments['cleaned_body'].str.split().str.len()

    before = len(comments)
    comments = comments[comments['word_count'] >= MIN_COMMENT_WORDS]
    print(f"  Removed {before - len(comments):,} comments under {MIN_COMMENT_WORDS} words")

    before = len(comments)
    comments = comments[comments['cleaned_body'].str.strip().str.len() > 0]
    print(f"  Removed {before - len(comments):,} comments with empty cleaned text")

    return comments


# ── MAIN ─────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print("=" * 80)
    print("FILTER + CLEAN TOP-LEVEL COMMENTS FOR bat_score>0 POSTS")
    print("=" * 80)

    target_post_ids = load_target_post_ids()

    all_filtered = []
    summary_rows = []

    for sub in SUBREDDITS:
        filtered = filter_subreddit(sub, target_post_ids)
        if filtered.empty:
            continue
        cleaned = clean_filtered_comments(filtered)
        cleaned['subreddit_source'] = sub
        all_filtered.append(cleaned)
        summary_rows.append({'subreddit': sub, 'comments_kept': len(cleaned)})

    master_comments = pd.concat(all_filtered, ignore_index=True) if all_filtered else pd.DataFrame()

    OUT_COLS = ['id', 'post_id', 'parent_id', 'link_id', 'author',
                'created_date', 'created_utc', 'year', 'month', 'year_month',
                'body', 'cleaned_body', 'word_count', 'score', 'permalink',
                'subreddit_source']
    master_comments = master_comments[[c for c in OUT_COLS if c in master_comments.columns]]
    master_comments.to_csv(OUTPUT_DIR + 'master_comments_filtered.csv', index=False)

    # Coverage check: how many target posts actually have >=1 surviving comment
    covered_posts = master_comments['post_id'].nunique() if not master_comments.empty else 0
    print("\n" + "─" * 60)
    print("SUMMARY")
    print("─" * 60)
    print(f"Total top-level comments kept: {len(master_comments):,}")
    print(f"Target posts (bat_score>0):    {len(target_post_ids):,}")
    print(f"Posts with ≥1 surviving comment: {covered_posts:,} "
          f"({covered_posts/len(target_post_ids)*100:.1f}%)")

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(OUTPUT_DIR + 'comment_filter_summary.csv', index=False)

    print(f"\n✓ master_comments_filtered.csv  →  {len(master_comments):,} comments")
    print(f"✓ comment_filter_summary.csv saved")

FILTER + CLEAN TOP-LEVEL COMMENTS FOR bat_score>0 POSTS
Target posts (bat_score>0): 12,237 of 12,237 total

Filtering comments for r/ciso…
  Detected format: ndjson
  Scanned 4,370 raw comments → kept 195 top-level comments on target posts
  ✓ Checkpoint saved: /Users/nadia/Desktop/redditRun_june/comment_data/processed_subreddits/filtered_raw_ciso_comments.csv
  Removed 0 duplicate comments
  Date window filter: kept 195 (removed 0)
  Removed 1 deleted/removed comments
  Removed 18 comments under 10 words
  Removed 0 comments with empty cleaned text

Filtering comments for r/cybersecurity…
  Detected format: ndjson
  Scanned 1,510,184 raw comments → kept 27,589 top-level comments on target posts
  ✓ Checkpoint saved: /Users/nadia/Desktop/redditRun_june/comment_data/processed_subreddits/filtered_raw_cybersecurity_comments.csv
  Removed 0 duplicate comments
  Date window filter: kept 27,588 (removed 1)
  Removed 877 deleted/removed comments
  Removed 2,898 comments under 10 words
  Remov

In [1]:
"""
check_vote_fields.py
Check whether raw comment ndjson records have separate ups/downs fields
beyond just 'score' (Reddit deprecated ups/downs for most API data since
~2016 -- ups typically mirrors score and downs is hardcoded to 0, but some
older Pushshift dumps have real values).

Usage:
    python check_vote_fields.py /path/to/raw_ciso_comments.ndjson
"""

import sys
import json

VOTE_RELATED_KEYS = ["score", "ups", "downs", "upvote_ratio", "controversiality"]


def main():
    if len(sys.argv) < 2:
        print("Usage: python check_vote_fields.py /path/to/file.ndjson")
        sys.exit(1)

    path = "/Users/nadia/Desktop/redditRun_june/comment_data/r_ciso_comments.jsonl"
    n_checked = 0
    found_keys = set()
    samples = []

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            n_checked += 1
            found_keys.update(k for k in VOTE_RELATED_KEYS if k in rec)

            if len(samples) < 5:
                sample = {k: rec.get(k) for k in VOTE_RELATED_KEYS if k in rec}
                if sample:
                    samples.append(sample)

            if n_checked >= 5000:
                break

    print(f"Checked {n_checked} records from {path}\n")
    print(f"Vote-related keys present anywhere: {sorted(found_keys)}\n")

    if "ups" in found_keys or "downs" in found_keys:
        print("Sample vote fields from first 5 records with any vote data:")
        for s in samples:
            print(" ", s)

        # Check if ups == score and downs == 0 everywhere sampled (the
        # deprecated/no-op pattern) vs genuinely differing values
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            mismatches = 0
            checked = 0
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if "ups" in rec and "score" in rec:
                    checked += 1
                    if rec["ups"] != rec["score"] or rec.get("downs", 0) != 0:
                        mismatches += 1
                if checked >= 2000:
                    break
        print(f"\nOf {checked} records checked: {mismatches} had ups != score "
              f"or downs != 0.")
        if mismatches == 0:
            print("→ Looks like the deprecated no-op pattern: 'ups' just mirrors "
                  "'score' and 'downs' is always 0. No extra signal beyond 'score'.")
        else:
            print("→ Some records have genuinely distinct ups/downs values -- "
                  "worth pulling these in separately.")
    else:
        print("No 'ups' or 'downs' fields found in this file -- only 'score' "
              "is available, which is the net (upvotes - downvotes) value.")


if __name__ == "__main__":
    main()

Checked 4370 records from /Users/nadia/Desktop/redditRun_june/comment_data/r_ciso_comments.jsonl

Vote-related keys present anywhere: ['controversiality', 'downs', 'score', 'ups']

Sample vote fields from first 5 records with any vote data:
  {'score': 2, 'ups': 2, 'controversiality': 0}
  {'score': 2, 'ups': 2, 'controversiality': 0}
  {'score': 3, 'ups': 3, 'controversiality': 0}
  {'score': 1, 'ups': 1, 'controversiality': 0}
  {'score': 1, 'ups': 1, 'controversiality': 0}

Of 2000 records checked: 0 had ups != score or downs != 0.
→ Looks like the deprecated no-op pattern: 'ups' just mirrors 'score' and 'downs' is always 0. No extra signal beyond 'score'.


In [2]:
"""
DIAGNOSTIC — EMPTY / SHORT COMMENTS CHECK
=============================================================
Checks two things:
  1. In the RAW checkpoint files (filtered_raw_<sub>_comments.csv,
     written BEFORE cleaning): how many comments are empty, deleted,
     removed, or very short? What's the length distribution?
  2. In the FINAL output (master_comments_filtered.csv, written
     AFTER clean_filtered_comments() should have removed all of
     the above): are any empty/short comments still present?
     If yes, that's the bug — clean_filtered_comments() is not
     doing what it's supposed to.

Read-only. Does not modify any pipeline files.
"""

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── CONFIGURATION ────────────────────────────────────────────────────────────
DATA_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'
PROCESSED_DATA_DIR = DATA_DIR + 'processed_subreddits/'
MASTER_FILE = DATA_DIR + 'master_comments_filtered.csv'
SUBREDDITS = ['ciso', 'cybersecurity', 'SecurityCareerAdvice', 'sysadmin', 'asknetsec']
MIN_COMMENT_WORDS = 10  # same threshold as pipeline, for cross-checking


def check_raw_checkpoints():
    print("=" * 80)
    print("PART 1: RAW CHECKPOINT FILES (filtered_raw_<sub>_comments.csv)")
    print("These are BEFORE cleaning — empty/deleted comments here are expected.")
    print("=" * 80)

    all_raw = []
    for sub in SUBREDDITS:
        path = PROCESSED_DATA_DIR + f'filtered_raw_{sub}_comments.csv'
        if not os.path.exists(path):
            print(f"\nr/{sub}: checkpoint not found at {path} — skipping")
            continue

        df = pd.read_csv(path, dtype={'body': str})
        df['subreddit_source'] = sub
        all_raw.append(df)

        total = len(df)
        body = df['body'].fillna('')
        n_nan = df['body'].isna().sum()
        n_literal_nan_str = (body.str.strip().str.lower() == 'nan').sum()
        n_empty_str = (body.str.strip() == '').sum()
        n_deleted = (body.str.strip() == '[deleted]').sum()
        n_removed = (body.str.strip() == '[removed]').sum()

        word_counts = body.str.split().str.len().fillna(0)

        print(f"\nr/{sub}: {total:,} rows")
        print(f"  body is NaN:              {n_nan:,}")
        print(f"  body is literal 'nan' str: {n_literal_nan_str:,}")
        print(f"  body is empty string:      {n_empty_str:,}")
        print(f"  body == '[deleted]':       {n_deleted:,}")
        print(f"  body == '[removed]':       {n_removed:,}")
        print(f"  min raw word count:        {word_counts.min():.0f}")
        print(f"  pct with < {MIN_COMMENT_WORDS} words:      "
              f"{(word_counts < MIN_COMMENT_WORDS).mean()*100:.1f}%")

    if not all_raw:
        print("\nNo raw checkpoint files found. Check PROCESSED_DATA_DIR path.")
        return None

    combined = pd.concat(all_raw, ignore_index=True)
    body = combined['body'].fillna('')
    word_counts = body.str.split().str.len().fillna(0)

    print("\n" + "─" * 60)
    print("COMBINED ACROSS ALL SUBREDDITS (raw, pre-clean)")
    print("─" * 60)
    print(f"Total raw top-level comments on target posts: {len(combined):,}")
    print(f"  Empty / NaN body:          {(body.str.strip() == '').sum() + combined['body'].isna().sum():,}")
    print(f"  [deleted]:                 {(body.str.strip() == '[deleted]').sum():,}")
    print(f"  [removed]:                 {(body.str.strip() == '[removed]').sum():,}")
    print(f"  Under {MIN_COMMENT_WORDS} words:              {(word_counts < MIN_COMMENT_WORDS).sum():,} "
          f"({(word_counts < MIN_COMMENT_WORDS).mean()*100:.1f}%)")
    print(f"  Zero words (empty/whitespace only): {(word_counts == 0).sum():,}")

    print("\nWord count distribution (raw body, before clean_text):")
    print(word_counts.describe())

    print("\nShortest 10 non-null raw bodies (to eyeball what 'short' looks like):")
    sample = combined.loc[body.str.strip() != '', ['id', 'post_id', 'subreddit_source', 'body']].copy()
    sample['wc'] = sample['body'].str.split().str.len()
    print(sample.sort_values('wc').head(10)[['subreddit_source', 'wc', 'body']].to_string(index=False))

    return combined


def check_final_output():
    print("\n" + "=" * 80)
    print("PART 2: FINAL OUTPUT (master_comments_filtered.csv)")
    print("These are AFTER clean_filtered_comments() — should have ZERO empty")
    print("comments and ZERO comments under MIN_COMMENT_WORDS. If not, that's the bug.")
    print("=" * 80)

    if not os.path.exists(MASTER_FILE):
        print(f"\nFinal file not found at {MASTER_FILE}")
        return

    df = pd.read_csv(MASTER_FILE, dtype={'cleaned_body': str, 'body': str})
    total = len(df)
    print(f"\nTotal rows in master_comments_filtered.csv: {total:,}")

    if 'cleaned_body' in df.columns:
        cb = df['cleaned_body'].fillna('')
        n_empty_cleaned = (cb.str.strip() == '').sum()
        print(f"  cleaned_body empty/whitespace only: {n_empty_cleaned:,} "
              f"({n_empty_cleaned/total*100:.1f}%)")
    else:
        print("  ⚠ 'cleaned_body' column not found in master file")

    if 'word_count' in df.columns:
        wc = df['word_count']
        n_zero = (wc == 0).sum()
        n_under_min = (wc < MIN_COMMENT_WORDS).sum()
        print(f"  word_count == 0:                    {n_zero:,} ({n_zero/total*100:.1f}%)")
        print(f"  word_count < {MIN_COMMENT_WORDS}:                   {n_under_min:,} "
              f"({n_under_min/total*100:.1f}%)")
        print(f"  min word_count in file:             {wc.min()}")
        print(f"\nword_count distribution in final file:")
        print(wc.describe())
    else:
        print("  ⚠ 'word_count' column not found in master file — recomputing from cleaned_body")
        if 'cleaned_body' in df.columns:
            wc = df['cleaned_body'].fillna('').str.split().str.len()
            print(f"  recomputed word_count == 0: {(wc == 0).sum():,}")
            print(f"  recomputed word_count < {MIN_COMMENT_WORDS}: {(wc < MIN_COMMENT_WORDS).sum():,}")

    # Show actual offending rows if any exist
    if 'word_count' in df.columns:
        offenders = df[df['word_count'] < MIN_COMMENT_WORDS]
        if len(offenders) > 0:
            print(f"\n⚠ FOUND {len(offenders):,} rows that violate MIN_COMMENT_WORDS filter.")
            print("Sample of offending rows:")
            cols = [c for c in ['id', 'post_id', 'subreddit_source', 'word_count', 'body', 'cleaned_body'] if c in df.columns]
            print(offenders[cols].head(15).to_string(index=False))
        else:
            print(f"\n✓ No rows found under MIN_COMMENT_WORDS — filter worked correctly in final file.")


if __name__ == '__main__':
    check_raw_checkpoints()
    check_final_output()
    print("\n" + "=" * 80)
    print("DONE")
    print("=" * 80)

PART 1: RAW CHECKPOINT FILES (filtered_raw_<sub>_comments.csv)
These are BEFORE cleaning — empty/deleted comments here are expected.

r/ciso: 195 rows
  body is NaN:              0
  body is literal 'nan' str: 0
  body is empty string:      0
  body == '[deleted]':       0
  body == '[removed]':       1
  min raw word count:        1
  pct with < 10 words:      10.3%

r/cybersecurity: 27,589 rows
  body is NaN:              0
  body is literal 'nan' str: 0
  body is empty string:      0
  body == '[deleted]':       413
  body == '[removed]':       464
  min raw word count:        1
  pct with < 10 words:      13.9%

r/SecurityCareerAdvice: 3,184 rows
  body is NaN:              0
  body is literal 'nan' str: 0
  body is empty string:      0
  body == '[deleted]':       29
  body == '[removed]':       39
  min raw word count:        1
  pct with < 10 words:      13.4%

r/sysadmin: 31,427 rows
  body is NaN:              0
  body is literal 'nan' str: 0
  body is empty string:      1
  b

In [3]:
"""
BREAKDOWN: ZERO-COMMENT POSTS BY SUBREDDIT
=============================================================
Of the 12,237 bat_score>0 posts, 8,756 have NO rows at all in
master_comments_filtered.csv (no top-level comments survived
filtering/cleaning for that post_id).

This breaks that 8,756 down by subreddit, and expresses it both
as a raw count and as a % of that subreddit's own bat_score>0 posts
(since subreddits differ hugely in size/culture — r/sysadmin is a
much bigger, chattier community than r/ciso, for example — so raw
counts alone would just reflect subreddit size, not engagement rate).

Reads:
    master_comments_filtered.csv     (has 'subreddit_source' per comment)
    bat_score_pos.csv                (12,237 bat_score>0 posts, needs a
                                       subreddit column OR a join to get one)
    master_posts_burnout_only.csv    (fallback source for subreddit if
                                       bat_score_pos.csv doesn't have it)

Writes:
    zero_comment_posts_by_subreddit.csv
"""

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATA_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'
POSTS_DIR = '/Users/nadia/Desktop/redditRun_june/'   # adjust if master_posts file lives elsewhere
COMMENTS_FILE = DATA_DIR + 'master_comments_filtered.csv'
BAT_FILE = DATA_DIR + 'bat_score_pos.csv'
MASTER_POSTS_FILE = POSTS_DIR + 'master_posts_burnout_only.csv'
OUTPUT_FILE = DATA_DIR + 'zero_comment_posts_by_subreddit.csv'


def get_bat_posts_with_subreddit():
    bat = pd.read_csv(BAT_FILE)
    bat['post_id'] = bat['post_id'].astype(str)

    # Case 1: bat_score_pos.csv already has a subreddit column
    sub_col = next((c for c in bat.columns if c.lower() in
                     ('subreddit', 'subreddit_source', 'sub')), None)
    if sub_col:
        print(f"Using subreddit column found directly in bat_score_pos.csv: '{sub_col}'")
        bat = bat.rename(columns={sub_col: 'subreddit'})
        return bat[['post_id', 'bat_score', 'subreddit']]

    # Case 2: need to join against master_posts_burnout_only.csv to get subreddit
    print("No subreddit column in bat_score_pos.csv — joining against "
          f"{MASTER_POSTS_FILE} to recover it.")
    posts = pd.read_csv(MASTER_POSTS_FILE)
    id_col = 'id' if 'id' in posts.columns else 'post_id'
    posts[id_col] = posts[id_col].astype(str)

    sub_col2 = next((c for c in posts.columns if c.lower() in
                      ('subreddit', 'subreddit_source', 'sub')), None)
    if sub_col2 is None:
        raise ValueError(
            "Could not find a subreddit column in either bat_score_pos.csv "
            f"or {MASTER_POSTS_FILE}. Check column names manually."
        )

    posts = posts.rename(columns={id_col: 'post_id', sub_col2: 'subreddit'})
    merged = bat.merge(posts[['post_id', 'subreddit']], on='post_id', how='left')

    n_missing = merged['subreddit'].isna().sum()
    if n_missing:
        print(f"⚠ {n_missing:,} posts ({n_missing/len(merged)*100:.1f}%) had no "
              "subreddit match after the join — check post_id overlap.")

    return merged[['post_id', 'bat_score', 'subreddit']]


def main():
    print("=" * 80)
    print("ZERO-COMMENT POSTS BY SUBREDDIT")
    print("=" * 80)

    bat = get_bat_posts_with_subreddit()
    comments = pd.read_csv(COMMENTS_FILE, usecols=['post_id'])
    comments['post_id'] = comments['post_id'].astype(str)

    posts_with_comments = set(comments['post_id'].unique())
    bat['has_comments'] = bat['post_id'].isin(posts_with_comments)

    print(f"\nTotal bat_score>0 posts: {len(bat):,}")
    print(f"Posts with >=1 surviving comment: {bat['has_comments'].sum():,}")
    print(f"Posts with ZERO surviving comments: {(~bat['has_comments']).sum():,}")

    summary = bat.groupby('subreddit').agg(
        total_bat_posts=('post_id', 'count'),
        posts_with_comments=('has_comments', 'sum'),
    ).reset_index()
    summary['posts_zero_comments'] = summary['total_bat_posts'] - summary['posts_with_comments']
    summary['pct_zero_comments'] = (summary['posts_zero_comments'] /
                                     summary['total_bat_posts'] * 100).round(1)
    summary['pct_with_comments'] = (summary['posts_with_comments'] /
                                     summary['total_bat_posts'] * 100).round(1)
    summary = summary.sort_values('total_bat_posts', ascending=False)

    print("\n" + "─" * 60)
    print("BREAKDOWN BY SUBREDDIT")
    print("─" * 60)
    print(summary.to_string(index=False))

    print("\n" + "─" * 60)
    print("SHARE OF THE 8,756 ZERO-COMMENT POSTS COMING FROM EACH SUBREDDIT")
    print("─" * 60)
    total_zero = summary['posts_zero_comments'].sum()
    summary['pct_of_all_zero_comment_posts'] = (summary['posts_zero_comments'] /
                                                  total_zero * 100).round(1)
    print(summary[['subreddit', 'posts_zero_comments', 'pct_of_all_zero_comment_posts']]
          .sort_values('posts_zero_comments', ascending=False).to_string(index=False))

    summary.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✓ Saved: {OUTPUT_FILE}")


if __name__ == '__main__':
    main()

ZERO-COMMENT POSTS BY SUBREDDIT
No subreddit column in bat_score_pos.csv — joining against /Users/nadia/Desktop/redditRun_june/master_posts_burnout_only.csv to recover it.

Total bat_score>0 posts: 12,237
Posts with >=1 surviving comment: 3,481
Posts with ZERO surviving comments: 8,756

────────────────────────────────────────────────────────────
BREAKDOWN BY SUBREDDIT
────────────────────────────────────────────────────────────
           subreddit  total_bat_posts  posts_with_comments  posts_zero_comments  pct_zero_comments  pct_with_comments
            sysadmin             9871                 1286                 8585               87.0               13.0
       cybersecurity             1626                 1520                  106                6.5               93.5
SecurityCareerAdvice              529                  470                   59               11.2               88.8
           AskNetsec              199                  193                    6                

In [6]:
"""
BREAKDOWN: HOW 54,935 COMMENTS SPREAD ACROSS 3,481 POSTS, BY SUBREDDIT
=============================================================
For the posts that DO have >=1 surviving comment, this shows,
per subreddit:
  - how many posts have comments
  - how many total comments they collectively have
  - the distribution (min/median/mean/max/std) of comment_count
    per post within that subreddit
  - top 5 most-commented posts per subreddit (to see who's driving
    the skew)

This directly answers "how does 54,935 spread over 3,481 posts,
broken down by subreddit" — i.e. is the concentration/skew uniform
across subreddits, or is one subreddit (e.g. sysadmin) doing most
of the heavy lifting?

Reads:
    master_comments_filtered.csv     (has subreddit_source per comment)
    bat_score_pos.csv                (bat_score>0 posts)

Writes:
    comment_spread_by_subreddit_summary.csv
    top_commented_posts_by_subreddit.csv
"""

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATA_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'
COMMENTS_FILE = DATA_DIR + 'master_comments_filtered.csv'
BAT_FILE = DATA_DIR + 'bat_score_pos.csv'
OUTPUT_SUMMARY = DATA_DIR + 'comment_spread_by_subreddit_summary.csv'
OUTPUT_TOP = DATA_DIR + 'top_commented_posts_by_subreddit.csv'
TOP_N_PER_SUB = 5


def main():
    print("=" * 80)
    print("HOW 54,935 COMMENTS SPREAD ACROSS 3,481 POSTS, BY SUBREDDIT")
    print("=" * 80)

    comments = pd.read_csv(COMMENTS_FILE)
    comments['post_id'] = comments['post_id'].astype(str)

    if 'subreddit_source' not in comments.columns:
        raise ValueError("Expected 'subreddit_source' column in "
                          "master_comments_filtered.csv — check the column name.")

    # comment_count per post, keeping subreddit_source (should be 1:1 with post_id)
    post_level = comments.groupby(['post_id', 'subreddit_source']).size().reset_index(name='comment_count')

    total_comments = post_level['comment_count'].sum()
    total_posts = post_level['post_id'].nunique()
    print(f"\nSanity check: {total_posts:,} posts, {total_comments:,} total comments "
          f"(should match 3,481 and 54,935)")

    # ── Per-subreddit summary ───────────────────────────────────────────────
    summary = post_level.groupby('subreddit_source')['comment_count'].agg(
        n_posts_with_comments='count',
        total_comments='sum',
        mean_comments_per_post='mean',
        median_comments_per_post='median',
        std_comments_per_post='std',
        max_comments_on_one_post='max',
    ).reset_index()

    summary['pct_of_total_comments'] = (summary['total_comments'] /
                                         total_comments * 100).round(1)
    summary['pct_of_total_posts_with_comments'] = (summary['n_posts_with_comments'] /
                                                     total_posts * 100).round(1)
    summary = summary.sort_values('total_comments', ascending=False)
    summary[['mean_comments_per_post', 'median_comments_per_post',
              'std_comments_per_post']] = summary[
        ['mean_comments_per_post', 'median_comments_per_post', 'std_comments_per_post']
    ].round(2)

    print("\n" + "─" * 60)
    print("PER-SUBREDDIT SUMMARY (posts with >=1 comment only)")
    print("─" * 60)
    print(summary.to_string(index=False))

    summary.to_csv(OUTPUT_SUMMARY, index=False)
    print(f"\n✓ Saved: {OUTPUT_SUMMARY}")

    # ── Top N most-commented posts per subreddit (shows the skew directly) ──
    top_posts = (post_level.sort_values('comment_count', ascending=False)
                 .groupby('subreddit_source')
                 .head(TOP_N_PER_SUB)
                 .sort_values(['subreddit_source', 'comment_count'], ascending=[True, False]))

    print("\n" + "─" * 60)
    print(f"TOP {TOP_N_PER_SUB} MOST-COMMENTED POSTS PER SUBREDDIT")
    print("─" * 60)
    print(top_posts.to_string(index=False))

    top_posts.to_csv(OUTPUT_TOP, index=False)
    print(f"\n✓ Saved: {OUTPUT_TOP}")

    # ── Overall skew check: how much of the total comes from top 10% of posts ─
    post_level_sorted = post_level.sort_values('comment_count', ascending=False)
    top_10pct_n = max(1, int(len(post_level_sorted) * 0.10))
    top_10pct_sum = post_level_sorted.head(top_10pct_n)['comment_count'].sum()
    print("\n" + "─" * 60)
    print("OVERALL CONCENTRATION CHECK")
    print("─" * 60)
    print(f"Top 10% most-commented posts ({top_10pct_n:,} posts) account for "
          f"{top_10pct_sum:,} of {total_comments:,} comments "
          f"({top_10pct_sum/total_comments*100:.1f}%)")


if __name__ == '__main__':
    main()

HOW 54,935 COMMENTS SPREAD ACROSS 3,481 POSTS, BY SUBREDDIT

Sanity check: 3,481 posts, 54,935 total comments (should match 3,481 and 54,935)

────────────────────────────────────────────────────────────
PER-SUBREDDIT SUMMARY (posts with >=1 comment only)
────────────────────────────────────────────────────────────
    subreddit_source  n_posts_with_comments  total_comments  mean_comments_per_post  median_comments_per_post  std_comments_per_post  max_comments_on_one_post  pct_of_total_comments  pct_of_total_posts_with_comments
            sysadmin                   1286           26652                   20.72                       9.0                  36.58                       583                   48.5                              36.9
       cybersecurity                   1520           23813                   15.67                       7.0                  23.51                       268                   43.3                              43.7
SecurityCareerAdvice               

In [5]:
"""
SAMPLE: 10 POSTS THAT HAVE SURVIVING TOP-LEVEL COMMENTS
=============================================================
Pulls 10 random posts from the 3,481 bat_score>0 posts that DO
have at least one row in master_comments_filtered.csv, and prints
the post detail plus its actual comments — so you can see what a
"successful" post/comment pair looks like next to the zero-comment
sample.

Reads:
    master_posts_burnout_only.csv    (post text/title/date)
    bat_score_pos.csv                (bat_score>0 posts)
    master_comments_filtered.csv     (comments, filtered/cleaned)

Writes:
    sample_posts_with_comments.csv
"""

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATA_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'
POSTS_DIR = '/Users/nadia/Desktop/redditRun_june/'
COMMENTS_FILE = DATA_DIR + 'master_comments_filtered.csv'
BAT_FILE = DATA_DIR + 'bat_score_pos.csv'
MASTER_POSTS_FILE = POSTS_DIR + 'master_posts_burnout_only.csv'
OUTPUT_FILE = DATA_DIR + 'sample_posts_with_comments.csv'
N_SAMPLE = 10
MAX_COMMENTS_SHOWN_PER_POST = 3   # don't dump all 50+ comments for a busy post
RANDOM_STATE = 42


def main():
    print("=" * 80)
    print(f"SAMPLE OF {N_SAMPLE} POSTS WITH SURVIVING COMMENTS")
    print("=" * 80)

    bat = pd.read_csv(BAT_FILE)
    bat['post_id'] = bat['post_id'].astype(str)

    comments = pd.read_csv(COMMENTS_FILE)
    comments['post_id'] = comments['post_id'].astype(str)
    posts_with_comments = set(comments['post_id'].unique())

    bat_with_comments = bat[bat['post_id'].isin(posts_with_comments)]
    print(f"Total posts with >=1 surviving comment available to sample from: "
          f"{len(bat_with_comments):,}")

    posts = pd.read_csv(MASTER_POSTS_FILE)
    id_col = 'id' if 'id' in posts.columns else 'post_id'
    posts[id_col] = posts[id_col].astype(str)
    posts = posts.rename(columns={id_col: 'post_id'})

    merged = posts[posts['post_id'].isin(bat_with_comments['post_id'])].merge(
        bat_with_comments[['post_id', 'bat_score']], on='post_id', how='left'
    )

    if merged.empty:
        print("\n⚠ No matching rows found after merging against "
              f"{MASTER_POSTS_FILE} — check post_id overlap.")
        return

    sample_posts = merged.sample(n=min(N_SAMPLE, len(merged)), random_state=RANDOM_STATE)

    preferred_post_cols = ['post_id', 'subreddit', 'bat_score', 'created_date', 'year_month',
                            'title', 'cleaned_text', 'text', 'selftext', 'score', 'num_comments']
    show_post_cols = [c for c in preferred_post_cols if c in sample_posts.columns]

    preferred_comment_cols = ['id', 'score', 'created_date', 'body', 'cleaned_body', 'word_count']
    show_comment_cols = [c for c in preferred_comment_cols if c in comments.columns]

    all_shown_comments = []

    print(f"\nShowing post columns: {show_post_cols}")
    print(f"Showing up to {MAX_COMMENTS_SHOWN_PER_POST} comments per post "
          f"(columns: {show_comment_cols})\n")

    for i, (_, row) in enumerate(sample_posts.iterrows(), 1):
        pid = row.get('post_id')
        print("=" * 70)
        print(f"[{i}] post_id: {pid}")
        for col in show_post_cols:
            if col == 'post_id':
                continue
            val = row.get(col)
            if isinstance(val, str) and len(val) > 300:
                val = val[:300] + '...'
            print(f"    {col}: {val}")

        post_comments = comments[comments['post_id'] == pid]
        total_c = len(post_comments)
        print(f"\n    → {total_c} total surviving comment(s) on this post. "
              f"Showing up to {MAX_COMMENTS_SHOWN_PER_POST}:")

        shown = post_comments.head(MAX_COMMENTS_SHOWN_PER_POST)
        for j, (_, crow) in enumerate(shown.iterrows(), 1):
            print(f"\n      comment {j}:")
            for ccol in show_comment_cols:
                val = crow.get(ccol)
                if isinstance(val, str) and len(val) > 250:
                    val = val[:250] + '...'
                print(f"        {ccol}: {val}")

        post_comments_out = post_comments.copy()
        post_comments_out['sample_post_rank'] = i
        all_shown_comments.append(post_comments_out)
        print()

    combined = pd.concat(all_shown_comments, ignore_index=True) if all_shown_comments else pd.DataFrame()
    combined.to_csv(OUTPUT_FILE, index=False)
    print(f"✓ Saved full comment sets for these {len(sample_posts)} posts: {OUTPUT_FILE}")


if __name__ == '__main__':
    main()

SAMPLE OF 10 POSTS WITH SURVIVING COMMENTS
Total posts with >=1 surviving comment available to sample from: 3,481

Showing post columns: ['post_id', 'subreddit', 'bat_score', 'created_date', 'year_month', 'title', 'cleaned_text', 'text', 'selftext', 'score', 'num_comments']
Showing up to 3 comments per post (columns: ['id', 'score', 'created_date', 'body', 'cleaned_body', 'word_count'])

[1] post_id: 8tteb5
    subreddit: sysadmin
    bat_score: 2
    created_date: 2018-06-25 19:31:00
    year_month: 2018-06
    title: Seeking advice
    cleaned_text: seeking advice so, i always keep seeing threads on here about people and their health, and i think i m reaching that point of concern for the first time as well, so i m seeking advice from those who have been in this field longer than i have or who ve experienced something similar. recently, i got a...
    text: Seeking advice
So, I always keep seeing threads on here about people and their health, and I think I’m reaching that point of co

In [4]:
"""
SAMPLE: 10 POSTS WITH ZERO SURVIVING TOP-LEVEL COMMENTS
=============================================================
Pulls 10 random posts from the 8,756 bat_score>0 posts that have
NO rows in master_comments_filtered.csv, and prints their full
post-level detail (subreddit, bat_score, date, title/text) so you
can eyeball what these actually look like — e.g. are they genuinely
uncommented posts, or very recent posts, or a particular subreddit/
post type?

Reads:
    master_posts_burnout_only.csv    (post text/title/date)
    bat_score_pos.csv                (bat_score>0 posts)
    master_comments_filtered.csv     (to know which post_ids have comments)

Writes:
    sample_zero_comment_posts.csv
"""

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATA_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'
POSTS_DIR = '/Users/nadia/Desktop/redditRun_june/'
COMMENTS_FILE = DATA_DIR + 'master_comments_filtered.csv'
BAT_FILE = DATA_DIR + 'bat_score_pos.csv'
MASTER_POSTS_FILE = POSTS_DIR + 'master_posts_burnout_only.csv'
OUTPUT_FILE = DATA_DIR + 'sample_zero_comment_posts.csv'
N_SAMPLE = 10
RANDOM_STATE = 42


def main():
    print("=" * 80)
    print(f"SAMPLE OF {N_SAMPLE} ZERO-COMMENT POSTS")
    print("=" * 80)

    bat = pd.read_csv(BAT_FILE)
    bat['post_id'] = bat['post_id'].astype(str)

    comments = pd.read_csv(COMMENTS_FILE, usecols=['post_id'])
    comments['post_id'] = comments['post_id'].astype(str)
    posts_with_comments = set(comments['post_id'].unique())

    bat['has_comments'] = bat['post_id'].isin(posts_with_comments)
    zero_comment_ids = bat.loc[~bat['has_comments'], 'post_id']

    print(f"Total zero-comment posts available to sample from: {len(zero_comment_ids):,}")

    posts = pd.read_csv(MASTER_POSTS_FILE)
    id_col = 'id' if 'id' in posts.columns else 'post_id'
    posts[id_col] = posts[id_col].astype(str)
    posts = posts.rename(columns={id_col: 'post_id'})

    zero_posts = posts[posts['post_id'].isin(zero_comment_ids)].merge(
        bat[['post_id', 'bat_score']], on='post_id', how='left'
    )

    if zero_posts.empty:
        print("\n⚠ No matching rows found after merging against "
              f"{MASTER_POSTS_FILE} — check that 'post_id' values line up "
              "between bat_score_pos.csv and the master posts file.")
        return

    sample = zero_posts.sample(n=min(N_SAMPLE, len(zero_posts)), random_state=RANDOM_STATE)

    # Print a readable view — show whichever columns actually exist
    preferred_cols = ['post_id', 'subreddit', 'bat_score', 'created_date', 'year_month',
                       'title', 'cleaned_text', 'text', 'selftext', 'score', 'num_comments']
    show_cols = [c for c in preferred_cols if c in sample.columns]

    print(f"\nShowing columns: {show_cols}\n")
    for i, (_, row) in enumerate(sample.iterrows(), 1):
        print("─" * 70)
        print(f"[{i}] post_id: {row.get('post_id')}")
        for col in show_cols:
            if col == 'post_id':
                continue
            val = row.get(col)
            if isinstance(val, str) and len(val) > 300:
                val = val[:300] + '...'
            print(f"    {col}: {val}")
    print("─" * 70)

    sample.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✓ Saved full sample: {OUTPUT_FILE}")


if __name__ == '__main__':
    main()

SAMPLE OF 10 ZERO-COMMENT POSTS
Total zero-comment posts available to sample from: 8,756

Showing columns: ['post_id', 'subreddit', 'bat_score', 'created_date', 'year_month', 'title', 'cleaned_text', 'text', 'selftext', 'score', 'num_comments']

──────────────────────────────────────────────────────────────────────
[1] post_id: 1c24iv5
    subreddit: sysadmin
    bat_score: 2
    created_date: 2024-04-12 09:07:07
    year_month: 2024-04
    title: Have been layed off, got a new job and now am scared of the changes coming
    cleaned_text: have been layed off, got a new job and now am scared of the changes coming basically the title. i have been working as a sysadmin in small companies teams for almost a decade now and have been responsible for the hellhole that s called m azure for the past years in my old company. during that time, ...
    text: Have been layed off, got a new job and now am scared of the changes coming
Basically the Title.  
I have been working as a Sysadmin in small 

In [7]:
"""
QUICK CHECK: ONE SPECIFIC POST_ID
=============================================================
Checks a single post_id against:
  1. filtered_raw_<sub>_comments.csv  (raw checkpoint, pre-clean —
     tells us if the comment scrape captured ANY comments for
     this post at all, top-level or not)
  2. master_comments_filtered.csv     (final output)

Edit POST_ID / SUBREDDIT below to check a different post.
"""

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'
PROCESSED_DATA_DIR = DATA_DIR + 'processed_subreddits/'
COMMENTS_FILE = DATA_DIR + 'master_comments_filtered.csv'

POST_ID = '1c24iv5'
SUBREDDIT = 'sysadmin'   # matches filtered_raw_sysadmin_comments.csv

raw_path = PROCESSED_DATA_DIR + f'filtered_raw_{SUBREDDIT}_comments.csv'
raw = pd.read_csv(raw_path, dtype={'body': str})
raw['post_id'] = raw['post_id'].astype(str)

raw_match = raw[raw['post_id'] == POST_ID]
print(f"Raw checkpoint ({raw_path}):")
print(f"  Rows found for post_id={POST_ID}: {len(raw_match)}")
if len(raw_match) > 0:
    print("\n  Raw rows found:")
    cols = [c for c in ['id', 'parent_id', 'link_id', 'body', 'score'] if c in raw_match.columns]
    print(raw_match[cols].to_string(index=False))
else:
    print("  → This post_id has ZERO rows in the raw checkpoint. This means either:")
    print("     (a) the comment scrape (r_sysadmin_comments.jsonl) never captured")
    print("         this post's comments at all, OR")
    print("     (b) all of this post's comments exist in the raw JSONL but are")
    print("         nested replies (parent_id starts with t1_, not t3_), so the")
    print("         stream_filter_comments() step excluded them before this")
    print("         checkpoint was even written.")
    print("     To tell these apart, we'd need to grep the raw .jsonl directly")
    print("     for this post_id in the link_id field, ignoring the t3_ check.")

final = pd.read_csv(COMMENTS_FILE, usecols=['post_id'])
final['post_id'] = final['post_id'].astype(str)
n_final = (final['post_id'] == POST_ID).sum()
print(f"\nFinal file (master_comments_filtered.csv):")
print(f"  Rows found for post_id={POST_ID}: {n_final}")

Raw checkpoint (/Users/nadia/Desktop/redditRun_june/comment_data/processed_subreddits/filtered_raw_sysadmin_comments.csv):
  Rows found for post_id=1c24iv5: 0
  → This post_id has ZERO rows in the raw checkpoint. This means either:
     (a) the comment scrape (r_sysadmin_comments.jsonl) never captured
         this post's comments at all, OR
     (b) all of this post's comments exist in the raw JSONL but are
         nested replies (parent_id starts with t1_, not t3_), so the
         stream_filter_comments() step excluded them before this
         checkpoint was even written.
     To tell these apart, we'd need to grep the raw .jsonl directly
     for this post_id in the link_id field, ignoring the t3_ check.

Final file (master_comments_filtered.csv):
  Rows found for post_id=1c24iv5: 0


In [8]:
"""
GREP RAW JSONL FOR A SPECIFIC POST_ID (IGNORING TOP-LEVEL FILTER)
=============================================================
The filtered_raw_<sub>_comments.csv checkpoint only contains
TOP-LEVEL comments (parent_id starts with t3_). If a post_id has
zero rows there, we don't yet know if that's because:
  (a) the raw scrape never captured any comments for this post, or
  (b) it captured comments, but they're all nested replies
      (parent_id starts with t1_) and got excluded before the
      checkpoint was written.

This script streams the RAW .jsonl file directly and checks EVERY
comment's link_id against the target post_id — regardless of
parent_id — to answer that question definitively.

Edit POST_ID / SUBREDDIT below to check a different post.
"""

import json
import warnings
warnings.filterwarnings('ignore')

INPUT_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'
POST_ID = '1c24iv5'
SUBREDDIT = 'sysadmin'
RAW_PATH = INPUT_DIR + f'r_{SUBREDDIT}_comments.jsonl'

print(f"Scanning {RAW_PATH} for link_id containing post_id={POST_ID} ...")
print("(This may take a bit for large files — streaming line by line.)\n")

matches = []
scanned = 0

with open(RAW_PATH, 'r', encoding='utf-8', errors='replace') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        scanned += 1
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        link_id = str(rec.get('link_id', ''))
        if link_id.replace('t3_', '', 1) == POST_ID:
            matches.append({
                'id': rec.get('id'),
                'parent_id': rec.get('parent_id'),
                'link_id': rec.get('link_id'),
                'body_preview': str(rec.get('body', ''))[:150],
                'created_utc': rec.get('created_utc'),
            })

print(f"Scanned {scanned:,} raw comment records in r/{SUBREDDIT}")
print(f"Matches found for post_id={POST_ID}: {len(matches)}\n")

if not matches:
    print("→ ZERO matches anywhere in the raw JSONL, at any nesting level.")
    print("  This confirms (a): the comment scrape never captured ANY comments")
    print(f"  for post {POST_ID}, despite Reddit showing 13 comments live today.")
    print("  This points to a SCRAPE COVERAGE GAP — the comment collection run")
    print("  either missed this post, was rate-limited, ran before some comments")
    print("  were posted, or this post fell outside whatever collection logic")
    print("  (e.g. time window, pagination limit) was used when scraping comments.")
else:
    n_top_level = sum(1 for m in matches if str(m['parent_id']).startswith('t3_'))
    n_nested = len(matches) - n_top_level
    print(f"→ Found {len(matches)} comments total: {n_top_level} top-level (t3_), "
          f"{n_nested} nested replies (t1_)")
    print("\nAll matches:")
    for m in matches:
        print(f"  id={m['id']}  parent_id={m['parent_id']}  "
              f"body_preview={m['body_preview']!r}")
    if n_top_level == 0 and n_nested > 0:
        print("\n→ This confirms (b): comments WERE scraped for this post, but every")
        print("  single one is a nested reply, not top-level. Your t3_-only filter")
        print("  is correctly excluding them by design, not failing.")

Scanning /Users/nadia/Desktop/redditRun_june/comment_data/r_sysadmin_comments.jsonl for link_id containing post_id=1c24iv5 ...
(This may take a bit for large files — streaming line by line.)

Scanned 4,385,478 raw comment records in r/sysadmin
Matches found for post_id=1c24iv5: 0

→ ZERO matches anywhere in the raw JSONL, at any nesting level.
  This confirms (a): the comment scrape never captured ANY comments
  for post 1c24iv5, despite Reddit showing 13 comments live today.
  This points to a SCRAPE COVERAGE GAP — the comment collection run
  either missed this post, was rate-limited, ran before some comments
  were posted, or this post fell outside whatever collection logic
  (e.g. time window, pagination limit) was used when scraping comments.


In [9]:
"""
QUANTIFY: SCRAPE GAP vs NESTED-ONLY vs FILTERED-OUT
ACROSS ALL 8,756 ZERO-COMMENT POSTS
=============================================================
Single-post grepping (previous script) doesn't scale to 8,756 posts —
that would mean re-scanning a multi-million-line JSONL per post.

Instead: ONE pass per subreddit's raw .jsonl builds two sets:
  - all_post_ids_any_comment: post_ids that appear as link_id AT ALL
    (any nesting level, any comment)
  - all_post_ids_top_level:   post_ids that have at least one TOP-LEVEL
    (t3_-parented) comment

Then, for every bat_score>0 post with ZERO rows in
master_comments_filtered.csv, classify it as:
  (A) SCRAPE GAP        — post_id not in all_post_ids_any_comment at all
  (B) NESTED-ONLY        — post_id in all_post_ids_any_comment but NOT
                            in all_post_ids_top_level (comments exist,
                            all are replies-to-replies)
  (C) FILTERED OUT       — post_id IS in all_post_ids_top_level (so it
                            had top-level comments), meaning they were
                            dropped later by cleaning (deleted/removed/
                            too short/date window) — this is the ONLY
                            category that reflects your cleaning logic
                            rather than scrape/structure limitations

This directly tells you what fraction of "71.6% zero comment" is a
data-collection issue (A), a structural Reddit-discussion-pattern
issue (B), or your own filtering choices (C).

Reads:
    r_<sub>_comments.jsonl        (raw, per subreddit)
    bat_score_pos.csv
    master_comments_filtered.csv
    master_posts_burnout_only.csv (for subreddit lookup)

Writes:
    scrape_gap_diagnosis_full.csv
"""

import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ───────────────────────────────────────────────────────────────────
INPUT_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'
POSTS_DIR = '/Users/nadia/Desktop/redditRun_june/'
COMMENTS_FILE = INPUT_DIR + 'master_comments_filtered.csv'
BAT_FILE = INPUT_DIR + 'bat_score_pos.csv'
MASTER_POSTS_FILE = POSTS_DIR + 'master_posts_burnout_only.csv'
OUTPUT_FILE = INPUT_DIR + 'scrape_gap_diagnosis_full.csv'

# master_posts subreddit value -> raw jsonl filename suffix
SUBREDDIT_MAP = {
    'sysadmin': 'sysadmin',
    'cybersecurity': 'cybersecurity',
    'SecurityCareerAdvice': 'SecurityCareerAdvice',
    'AskNetsec': 'asknetsec',
    'asknetsec': 'asknetsec',
    'ciso': 'ciso',
    'CISO': 'ciso',
}


def scan_subreddit_jsonl(subreddit_suffix):
    """One pass: build set of post_ids with ANY comment, and set with
    at least one TOP-LEVEL comment."""
    path = INPUT_DIR + f'r_{subreddit_suffix}_comments.jsonl'
    any_comment = set()
    top_level = set()
    scanned = 0

    print(f"  Scanning {path} ...")
    try:
        with open(path, 'r', encoding='utf-8', errors='replace') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                scanned += 1
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue

                link_id = str(rec.get('link_id', ''))
                if not link_id:
                    continue
                pid = link_id.replace('t3_', '', 1)
                any_comment.add(pid)

                parent_id = str(rec.get('parent_id', ''))
                if parent_id.startswith('t3_'):
                    top_level.add(pid)
    except FileNotFoundError:
        print(f"  ⚠ File not found: {path}")
        return None, None, 0

    print(f"  Scanned {scanned:,} comments → {len(any_comment):,} distinct posts "
          f"with any comment, {len(top_level):,} with >=1 top-level comment")
    return any_comment, top_level, scanned


def main():
    print("=" * 80)
    print("QUANTIFYING SCRAPE GAP vs NESTED-ONLY vs FILTERED-OUT")
    print("=" * 80)

    # ── Build zero-comment target set with subreddit info ──────────────────
    bat = pd.read_csv(BAT_FILE, usecols=['post_id', 'bat_score'])
    bat['post_id'] = bat['post_id'].astype(str)

    posts = pd.read_csv(MASTER_POSTS_FILE)
    id_col = 'id' if 'id' in posts.columns else 'post_id'
    posts[id_col] = posts[id_col].astype(str)
    posts = posts.rename(columns={id_col: 'post_id'})
    sub_col = next((c for c in posts.columns if c.lower() in
                     ('subreddit', 'subreddit_source', 'sub')), None)
    posts = posts.rename(columns={sub_col: 'subreddit'})

    bat = bat.merge(posts[['post_id', 'subreddit']], on='post_id', how='left')

    comments = pd.read_csv(COMMENTS_FILE, usecols=['post_id'])
    comments['post_id'] = comments['post_id'].astype(str)
    posts_with_final_comments = set(comments['post_id'].unique())

    bat['has_final_comments'] = bat['post_id'].isin(posts_with_final_comments)
    zero_comment_posts = bat[~bat['has_final_comments']].copy()

    print(f"\nTotal zero-comment posts to diagnose: {len(zero_comment_posts):,}")

    # ── Scan each subreddit's raw JSONL once ────────────────────────────────
    subreddits_needed = zero_comment_posts['subreddit'].dropna().unique()
    print(f"\nSubreddits to scan: {list(subreddits_needed)}\n")

    any_comment_sets = {}
    top_level_sets = {}
    for sub in subreddits_needed:
        suffix = SUBREDDIT_MAP.get(sub, sub)
        print(f"r/{sub} (file suffix: {suffix}):")
        any_c, top_l, n = scan_subreddit_jsonl(suffix)
        any_comment_sets[sub] = any_c or set()
        top_level_sets[sub] = top_l or set()
        print()

    # ── Classify every zero-comment post ────────────────────────────────────
    def classify(row):
        sub = row['subreddit']
        pid = row['post_id']
        any_set = any_comment_sets.get(sub, set())
        top_set = top_level_sets.get(sub, set())

        if pid not in any_set:
            return 'A: SCRAPE GAP'
        elif pid not in top_set:
            return 'B: NESTED-ONLY (comments exist, none top-level)'
        else:
            return 'C: FILTERED OUT (had top-level comments, dropped by cleaning)'

    zero_comment_posts['diagnosis'] = zero_comment_posts.apply(classify, axis=1)

    print("=" * 80)
    print("FULL DIAGNOSIS ACROSS ALL ZERO-COMMENT POSTS")
    print("=" * 80)
    counts = zero_comment_posts['diagnosis'].value_counts()
    pct = (counts / len(zero_comment_posts) * 100).round(1)
    summary = pd.DataFrame({'count': counts, 'pct_of_zero_comment_posts': pct})
    print(summary)

    print("\n" + "─" * 60)
    print("BREAKDOWN BY SUBREDDIT")
    print("─" * 60)
    by_sub = zero_comment_posts.groupby(['subreddit', 'diagnosis']).size().unstack(fill_value=0)
    print(by_sub)

    zero_comment_posts.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✓ Saved: {OUTPUT_FILE}")


if __name__ == '__main__':
    main()

QUANTIFYING SCRAPE GAP vs NESTED-ONLY vs FILTERED-OUT

Total zero-comment posts to diagnose: 8,756

Subreddits to scan: ['cybersecurity', 'SecurityCareerAdvice', 'sysadmin', 'AskNetsec']

r/cybersecurity (file suffix: cybersecurity):
  Scanning /Users/nadia/Desktop/redditRun_june/comment_data/r_cybersecurity_comments.jsonl ...
  Scanned 1,510,184 comments → 155,979 distinct posts with any comment, 155,978 with >=1 top-level comment

r/SecurityCareerAdvice (file suffix: SecurityCareerAdvice):
  Scanning /Users/nadia/Desktop/redditRun_june/comment_data/r_SecurityCareerAdvice_comments.jsonl ...
  Scanned 79,568 comments → 7,780 distinct posts with any comment, 7,778 with >=1 top-level comment

r/sysadmin (file suffix: sysadmin):
  Scanning /Users/nadia/Desktop/redditRun_june/comment_data/r_sysadmin_comments.jsonl ...
  Scanned 4,385,478 comments → 211,089 distinct posts with any comment, 211,084 with >=1 top-level comment

r/AskNetsec (file suffix: asknetsec):
  Scanning /Users/nadia/Desk

In [10]:
"""
QUICK CHECK: WHAT DATE RANGE DO THE RAW COMMENT FILES ACTUALLY COVER?
=============================================================
Pushshift lost live access to Reddit's data in May 2023. Depending on
which dump/mirror was used, comment data may have a hard cutoff around
then (or some other date), meaning comments made after that date simply
don't exist in the file — regardless of scrape quality.

This scans each r_<sub>_comments.jsonl once and reports the min/max
created_utc found, plus a year-by-year comment count, so you can see
directly whether there's a cliff-edge cutoff (evidence of a frozen
Pushshift snapshot) or a gradual taper (evidence of something else).

Reads:
    r_<sub>_comments.jsonl   (raw, per subreddit)

Prints results only — this is a fast read-only diagnostic.
"""

import json
from datetime import datetime, timezone
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

INPUT_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'
SUBREDDITS = ['ciso', 'cybersecurity', 'SecurityCareerAdvice', 'sysadmin', 'asknetsec']


def scan_date_range(subreddit):
    path = INPUT_DIR + f'r_{subreddit}_comments.jsonl'
    min_ts, max_ts = None, None
    year_counts = Counter()
    scanned = 0

    try:
        with open(path, 'r', encoding='utf-8', errors='replace') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                scanned += 1
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue

                ts = rec.get('created_utc')
                if ts is None:
                    continue
                try:
                    ts = float(ts)
                except (TypeError, ValueError):
                    continue

                if min_ts is None or ts < min_ts:
                    min_ts = ts
                if max_ts is None or ts > max_ts:
                    max_ts = ts

                year = datetime.fromtimestamp(ts, tz=timezone.utc).year
                year_counts[year] += 1
    except FileNotFoundError:
        print(f"  ⚠ File not found: {path}")
        return

    min_date = datetime.fromtimestamp(min_ts, tz=timezone.utc).date() if min_ts else None
    max_date = datetime.fromtimestamp(max_ts, tz=timezone.utc).date() if max_ts else None

    print(f"r/{subreddit}: {scanned:,} raw comments scanned")
    print(f"  Date range: {min_date}  →  {max_date}")
    print(f"  Comments by year:")
    for yr in sorted(year_counts):
        print(f"    {yr}: {year_counts[yr]:,}")
    print()


if __name__ == '__main__':
    print("=" * 80)
    print("RAW COMMENT FILE DATE COVERAGE CHECK")
    print("=" * 80)
    for sub in SUBREDDITS:
        scan_date_range(sub)

RAW COMMENT FILE DATE COVERAGE CHECK
r/ciso: 4,370 raw comments scanned
  Date range: 2015-10-28  →  2026-04-28
  Comments by year:
    2015: 1
    2016: 6
    2017: 8
    2018: 23
    2019: 67
    2020: 165
    2021: 400
    2022: 32
    2023: 127
    2024: 663
    2025: 1,768
    2026: 1,110

r/cybersecurity: 1,510,184 raw comments scanned
  Date range: 2013-02-27  →  2026-04-26
  Comments by year:
    2013: 15
    2014: 119
    2015: 1,172
    2016: 2,261
    2017: 5,577
    2018: 17,692
    2019: 51,912
    2020: 105,064
    2021: 154,234
    2022: 203,756
    2023: 252,474
    2024: 300,666
    2025: 309,022
    2026: 106,220

r/SecurityCareerAdvice: 79,568 raw comments scanned
  Date range: 2018-12-20  →  2026-04-28
  Comments by year:
    2018: 139
    2019: 1,884
    2020: 2,638
    2021: 4,782
    2022: 5,643
    2023: 6,790
    2024: 10,177
    2025: 38,862
    2026: 8,653

r/sysadmin: 4,385,478 raw comments scanned
  Date range: 2009-03-12  →  2019-04-02
  Comments by year:


In [1]:
"""
HEADER CHECK — SCHEMA OVERVIEW FOR ALL CSVs IN A DIRECTORY
==============================================================
Prints columns, dtypes, row count, and a couple of sample values for
every .csv file found, so you can confirm key names (post_id vs id,
subreddit, date columns, etc.) before merging anything.

Usage:
    python3 check_csv_headers.py

Edit DIRS_TO_CHECK below to point at wherever your CSVs live —
defaults to checking both your main data folder and comment_data/.
"""

import pandas as pd
import glob
import os
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ───────────────────────────────────────────────────────────────────
DIRS_TO_CHECK = [
    # '/Users/nadia/Desktop/redditRun_june/',
    '/Users/nadia/Desktop/redditRun_june/comment_data/'
]

N_SAMPLE_ROWS = 2   # how many sample values to show per column


def check_file(filepath):
    filename = os.path.basename(filepath)
    print("=" * 80)
    print(filename)
    print("=" * 80)

    try:
        # Read just the header first (fast, even for huge files)
        header_df = pd.read_csv(filepath, nrows=0)
        n_cols = len(header_df.columns)

        # Accurate row count that respects quoted multiline fields
        # (raw newline counting overcounts when text/body columns contain
        # embedded newlines inside quoted CSV cells — same trap as `wc -l`)
        first_col = header_df.columns[0]
        n_rows = len(pd.read_csv(filepath, usecols=[first_col]))

        print(f"Rows: {n_rows:,}   Columns: {n_cols}")
        print("-" * 80)

        # Read a small sample for dtypes + example values
        sample = pd.read_csv(filepath, nrows=max(N_SAMPLE_ROWS, 100))

        for col in header_df.columns:
            dtype = sample[col].dtype if col in sample.columns else '?'
            examples = sample[col].dropna().head(N_SAMPLE_ROWS).tolist() if col in sample.columns else []
            print(f"  {col:35s} dtype={str(dtype):10s} e.g. {examples}")

    except Exception as e:
        print(f"  ⚠ Could not read file: {e}")

    print()


if __name__ == '__main__':
    all_csvs = []
    for d in DIRS_TO_CHECK:
        if os.path.isdir(d):
            found = sorted(glob.glob(os.path.join(d, '*.csv')))
            all_csvs.extend(found)
        else:
            print(f"⚠ Directory not found, skipping: {d}")

    all_csvs = sorted(set(all_csvs))
    print(f"Found {len(all_csvs)} CSV file(s) across {len(DIRS_TO_CHECK)} director{'y' if len(DIRS_TO_CHECK)==1 else 'ies'}\n")

    for filepath in all_csvs:
        check_file(filepath)

    print("=" * 80)
    print("DONE")
    print("=" * 80)

Found 8 CSV file(s) across 1 directory

bat_posts_results_final_patched.csv
Rows: 144,652   Columns: 16
--------------------------------------------------------------------------------
  row_type                            dtype=object     e.g. ['post', 'post']
  post_id                             dtype=object     e.g. ['cm8cbe', 'b88bq5']
  comment_id                          dtype=float64    e.g. []
  text                                dtype=object     e.g. ["Armoring yourself with web presence DLP solution\nHi fellows,\n\nI'm working for a mid-size e-commerce company, and recently heard a lot about attacks coming from the  3rd parties that are load in the website. I decided to take a quick research and came up with a few solutions that seem to address this issue.\n\nBefore I continue with the process, I wanted to ask here - has anyone of you guys taken some time to search for a solution in this area? And if you got there - what is the price rage that you received for such solution